In [1]:
!pwd

/home/jovyan/Deepfake_Low Resource_Low Resolution


In [2]:
%pip install torch scikit-learn pandas pillow tqdm numpy

  Using cached torch-2.13.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (38 kB)
  Using cached scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached pandas-3.0.3-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached pillow-12.3.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached filelock-3.29.7-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-83.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.0.3.0-py2.py3-none-any.whl.metadata (17 kB)
  Using cached cuda_bindings-13.3.1-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86

In [4]:
# Cleaning for corrupt images from the df40 dataset
import os
from pathlib import Path
import pandas as pd
from PIL import Image, ImageFile, UnidentifiedImageError
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

manifest_path = Path("outputs_df40/dataset_summary/df40_manifest_all.csv")
out_dir = manifest_path.parent

df = pd.read_csv(manifest_path)

# Find the image path column
path_col = None
for c in ["image_path", "frame_path", "path", "filepath", "file_path"]:
    if c in df.columns:
        path_col = c
        break

if path_col is None:
    raise ValueError(f"Could not find image path column. Columns are: {df.columns.tolist()}")

print("Using path column:", path_col)
print("Rows before checking:", len(df))

good_rows = []
bad_rows = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Checking images"):
    p = str(row[path_col])

    reason = None
    try:
        if not os.path.exists(p):
            reason = "missing_file"
        elif os.path.getsize(p) == 0:
            reason = "empty_file"
        else:
            # verify() checks file integrity, then reopen to make sure convert works
            with Image.open(p) as img:
                img.verify()
            with Image.open(p) as img:
                img.convert("RGB")
    except Exception as e:
        reason = f"{type(e).__name__}: {e}"

    if reason is None:
        good_rows.append(idx)
    else:
        bad = row.to_dict()
        bad["bad_image_reason"] = reason
        bad["file_size_bytes"] = os.path.getsize(p) if os.path.exists(p) else None
        bad_rows.append(bad)

clean_df = df.loc[good_rows].reset_index(drop=True)
bad_df = pd.DataFrame(bad_rows)

clean_manifest = out_dir / "df40_manifest_all_clean.csv"
bad_csv = out_dir / "bad_or_corrupt_images.csv"

clean_df.to_csv(clean_manifest, index=False)
bad_df.to_csv(bad_csv, index=False)

# Also save clean split CSVs
for split in ["train", "val", "test"]:
    split_df = clean_df[clean_df["split"] == split].reset_index(drop=True)
    split_df.to_csv(out_dir / f"df40_manifest_{split}_clean.csv", index=False)

print("Rows after cleaning:", len(clean_df))
print("Bad/corrupt images:", len(bad_df))
print("Saved clean manifest to:", clean_manifest)
print("Saved bad image list to:", bad_csv)

print("\nClean split counts:")
print(pd.crosstab(clean_df["split"], clean_df["label_name"]))

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_780/1622915467.py:12: DtypeWarning: Columns (0: unclear_reason) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(manifest_path)


Using path column: frame_path
Rows before checking: 468574


Checking images: 100%|██████████| 468574/468574 [4:22:39<00:00, 29.73it/s]  


Rows after cleaning: 468569
Bad/corrupt images: 5
Saved clean manifest to: outputs_df40/dataset_summary/df40_manifest_all_clean.csv
Saved bad image list to: outputs_df40/dataset_summary/bad_or_corrupt_images.csv

Clean split counts:
label_name    fake   real
split                    
test         47814    994
train       361226  11539
val          45462   1534


In [3]:
#DF40 implementation
!python grfn_csf_grfn_only_v6.py \
  --metadata_csv "outputs_df40/df40_manifest_all_clean.csv" \
  --out_dir "outputs_grfn_df40_indomain_auto_balance" \
  --dataset_name "DF40" \
  --epochs 15 \
  --batch_size 16 \
  --image_size 224 \
  --frames_per_video 32 \
  --balance_strategy auto \
  --num_workers 2 \
  --amp


GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_grfn_df40_indomain_auto_balance
Metadata CSV: outputs_df40/df40_manifest_all_clean.csv
Streams: rgb,fingerprint,jpeg,freq

Metadata loaded
Metadata CSV: outputs_df40/df40_manifest_all_clean.csv
Path column: frame_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 468569
train: frames=372765, units=15155, real=11539, fake=361226
val: frames=46996, units=1893, real=1534, fake=45462
test: frames=48808, units=1895, real=994, fake=47814
Saved label audit samples to: outputs_grfn_df40_indomain_auto_balance/label_audit_samples.csv

Sampling frames per unit
Sample train: 100%|█████████████████████| 15155/15155 [06:00<00:00, 42.04unit/s]
Sample train: frames=340460, units=15155, real=4666, fake=335794
Sample val: 100%|████████████████████████| 1893/1893 [00:06<00:00, 272.77unit/s]
Sample val: frames=43200, units=1893, real=741, fake

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Train epoch 5: 100%|█| 21278/21278 [2:36:10<00:00,  2.27it/s, cls=0.0079, cons=0
Predict val: 100%|██████████████████████████| 2700/2700 [25:45<00:00,  1.75it/s]
Validation-tuned decision threshold: 1.00
New best val frame AUCROC: 0.9743

Validation metrics:
Level  Decision Threshold  Accuracy  Balanced Accuracy    AUPRC   AUCROC  EER (Equal Error Rate)  Precision (0)  Recall (0)   F1 (0)  Precision (1)  Recall (1)   F1 (1)  Uncertain Rate (%)  Predicted Real Count  Predicted Fake Count
Frame            0.999448  0.907963           0.906106 0.999494 0.974322                0.094585       0.146448    0.904184 0.252069       0.998162    0.908029 0.950964            0.645833                  4575                 38625
Video            0.999448  0.821447           0.891944 0.999597 0.998123                0.015329       0.478328    0.996774 0.646444       0.999198    0.787113 0.880565            0.158479                   646                  1247
Train epoch 6: 100%|█| 21278/21278 [2:37:5

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Train epoch 13: 100%|█| 21278/21278 [2:26:14<00:00,  2.42it/s, cls=0.0020, cons=
Predict val: 100%|██████████████████████████| 2700/2700 [26:51<00:00,  1.67it/s]
Validation-tuned decision threshold: 1.00
New best val frame AUCROC: 0.9860

Validation metrics:
Level  Decision Threshold  Accuracy  Balanced Accuracy    AUPRC   AUCROC  EER (Equal Error Rate)  Precision (0)  Recall (0)   F1 (0)  Precision (1)  Recall (1)   F1 (1)  Uncertain Rate (%)  Predicted Real Count  Predicted Fake Count
Frame            0.999958  0.923657           0.936632 0.999729 0.985986                0.064761       0.177554    0.950067 0.299193       0.999057    0.923196 0.959630            0.106481                  3965                 39235
Video            0.999958  0.826202           0.896083 0.999874 0.999376                0.009577       0.485133    1.000000 0.653319       1.000000    0.792167 0.884032            0.052826                   639                  1254
Train epoch 14: 100%|█| 21278/21278 [2:33:

In [ ]:
print('test')

In [ ]:
!python grfn_csf_grfn_only_v6.py \
  --metadata_csv /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv \
  --out_dir outputs_grfn_only_v6_balanced \
  --epochs 15 \
  --image_size 224 \
  --frames_per_video 32 \
  --batch_size 16 \
  --num_workers 2 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --feat_dim 128 \
  --streams rgb,fingerprint,jpeg,freq \
  --threshold_metric balanced_accuracy \
  --balance_strategy auto \
  --score_direction auto_val \
  --stream_logit_residual_weight 0.25 \
  --amp


GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/outputs_grfn_only_v6_balanced
Metadata CSV: /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv
Streams: rgb,fingerprint,jpeg,freq

Metadata loaded
Metadata CSV: /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv
Path column: frame_path
Label column: label
Split column: split
Group column: video_id
Usable frames: 55953
train: frames=44756, units=5600, real=6400, fake=38356
val: frames=5600, units=700, real=800, fake=4800
test: frames=5597, units=700, real=800, fake=4797
Saved label audit samples to: outputs_grfn_only_v6_balanced/label_audit_samples.csv

Sampling frames per unit
Sample train: 100%|██████████████████████| 5600/5600 [00:17<00:00, 317.37unit/s]
Sample train: frames=44756, units=5600, real=6400, fake=38356
Sample val: 100%|█████████████████████████| 700/700

In [3]:
!python grfn_csf_grfn_only_v6.py \
  --metadata_csv /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv \
  --out_dir outputs_grfn_only_v6_balanced \
  --epochs 30 \
  --image_size 224 \
  --frames_per_video 32 \
  --batch_size 16 \
  --num_workers 2 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  # --feat_dim 128 \
  --streams rgb,fingerprint,jpeg,freq \
  --threshold_metric balanced_accuracy \
  --balance_strategy auto \
  --score_direction auto_val \
  --stream_logit_residual_weight 0.25 \
  --amp


GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_grfn_only_v6_balanced
Metadata CSV: /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv
Streams: rgb,fingerprint,jpeg,freq

Metadata loaded
Metadata CSV: /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv
Path column: frame_path
Label column: label
Split column: split
Group column: video_id
Usable frames: 55953
train: frames=44756, units=5600, real=6400, fake=38356
val: frames=5600, units=700, real=800, fake=4800
test: frames=5597, units=700, real=800, fake=4797
Saved label audit samples to: outputs_grfn_only_v6_balanced/label_audit_samples.csv

Sampling frames per unit
Sample train: 100%|██████████████████████| 5600/5600 [00:18<00:00, 306.38unit/s]
Sample train: frames=44756, units=5600, real=6400, fake=38356
Sample val: 100%|█████████████████████████| 700/700 [00:00